# Домашнее задание — Занятие 45

## Свой LLM-агент на локальном стеке

**Оценка:** 100 баллов
**Примерное время:** 1-2 часа
**Дедлайн:** до следующего занятия

---

### Что нужно сделать

Собрать у себя **локально** тот же стек что мы проходили на уроке (Ollama + LangChain + Langfuse), но с собственным промптом и собственным агентом. Один из инструментов агента должен реально ходить во внешний API.

### Структура домашки

| № | Задание | Баллы |
|---|---|---|
| 0 | Поднять локальный стек (проверочные скриншоты) | 20 |
| 1 | Свой промпт через ChatPromptTemplate | 20 |
| 2 | Свой агент с 2-3 tools | 40 |
| 3 | Теоретические вопросы | 20 |

### Что сдавать

1. **Этот ноутбук** (`Homework_45_<фамилия>.ipynb`) — все ячейки должны быть выполнены, вывод сохранён
2. **Скриншоты** (приложить отдельно или вставить в markdown-ячейки):
   - `ollama list` в терминале
   - `docker compose ps` в папке langfuse (все 6 контейнеров Up)
   - Страница Langfuse UI со списком трейсов от твоего агента (минимум 5 трейсов)
   - Развёрнутый trace агента с видимыми tool calls

### Правила

- **Никакого копирования** кода из практики в задания 1 и 2. Промпт и tools должны быть твоими.
- **Нельзя** использовать OpenAI / Claude / Gemini API — только локальная Gemma через Ollama.
- Казахстанские источники данных в tools приветствуются, но не обязательны.

---


## Задание 0 — Локальный стек (20 баллов)

Подними на своей машине три сервиса:

1. **Ollama** + модель `gemma4:e4b`
2. **Langfuse self-hosted** через `docker compose` (следуй инструкции с урока)
3. **Python окружение** с пакетами: `langchain-ollama`, `langfuse>=3.14,<4.0`, `langchain-core`, `openai`, `requests`

**Сдача этого задания:**
- Скриншот терминала с `ollama list` (должна быть видна gemma4:e4b)
- Скриншот `docker compose ps` (6 контейнеров langfuse в статусе Up)
- Скриншот главной страницы Langfuse UI (http://localhost:3000) после логина

Вставь скриншоты в markdown-ячейку ниже (`![](path/to/screenshot.png)`) или приложи отдельно архивом при сдаче.

### Проверка что всё живо


In [3]:
# Установка пакетов — если ещё не ставил
%pip install -q --upgrade \
    "langchain-ollama>=0.3,<1.0" \
    "langchain-core>=0.3,<1.0" \
    "langchain-community>=0.3,<1.0" \
    "langfuse>=3.14,<4.0" \
    "openai>=1.50" \
    "requests>=2.31"

# ВАЖНО: после установки перезапусти kernel (Kernel → Restart) и выполняй дальше


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import requests
from langfuse import Langfuse, get_client

# 1. Настройка ключей
PK = "pk-lf-c8abc57f-8af3-4431-9817-c48993b13f66"
SK = "sk-lf-19edde66-d2f2-4d37-9dc3-e01bd31c8592"
HOST = "http://localhost:3000"

os.environ["LANGFUSE_PUBLIC_KEY"] = PK
os.environ["LANGFUSE_SECRET_KEY"] = SK
os.environ["LANGFUSE_HOST"] = HOST

print("--- Начинаю проверку стека ---")

# 2. Проверка Ollama
try:
    r_ollama = requests.get("http://localhost:11434/api/tags", timeout=5)
    if r_ollama.status_code == 200:
        models = [m["name"] for m in r_ollama.json().get("models", [])]
        
        target_model = "gemma4:e4b" 
        if any(target_model in m for m in models):
            print(f"✓ Ollama запущена, модель {target_model} найдена.")
        else:
            print(f"⚠️ Ollama работает, но модель {target_model} не найдена. Доступные: {models}")
    else:
        print("❌ Ollama ответила с ошибкой.")
except Exception as e:
    print(f"❌ Ollama не отвечает. Убедись, что приложение Ollama запущено. Ошибка: {e}")

# 3. Проверка Langfuse
try:
    lf_client = Langfuse(public_key=PK, secret_key=SK, host=HOST)
    
    if lf_client.auth_check():
        print("Langfuse поднят и ключи верные.")
    else:
        print("Langfuse: Ошибка авторизации. Проверь ключи в настройках проекта.")
except Exception as e:
    print(f"Не удалось связаться с Langfuse (Docker). Проверь docker compose ps. Ошибка: {e}")

print("\n✓ Задание 0 выполнено — стек готов к работе")

--- Начинаю проверку стека ---
✓ Ollama запущена, модель gemma4:e4b найдена.
Langfuse поднят и ключи верные.

✓ Задание 0 выполнено — стек готов к работе


### Сюда вставь скриншоты

```
[ollama list screenshot]
[docker compose ps screenshot]
[langfuse UI screenshot]
```


---

## Задание 1 — Свой промпт через ChatPromptTemplate (20 баллов)

Напиши свой `ChatPromptTemplate` и chain на Gemma. **Не sentiment analysis** (это было в практике) — придумай что-то своё.

### Варианты (выбери один или предложи свой)

- **Суммаризатор новостей** — вход: текст статьи, выход: 2-3 предложения резюме
- **Переводчик** с русского на казахский (или наоборот) с обязательным объяснением идиом
- **Генератор тестовых вопросов** — вход: тема, выход: 3 вопроса с вариантами ответов
- **Анализатор резюме** — вход: текст резюме, выход: сильные/слабые стороны
- **Генератор идей** — вход: область (например "казахстанский стартап"), выход: 3 идеи с кратким обоснованием

### Требования

1. Используй `ChatPromptTemplate.from_messages([...])` с ролями `system` и `human`
2. Минимум **2 переменных** в шаблоне (например `{topic}` и `{style}`, или `{source_lang}` и `{text}`)
3. Собери chain через LCEL: `prompt | llm | StrOutputParser()`
4. **Подключи Langfuse callback** — все вызовы должны попадать в трейсы
5. Запусти chain на **минимум 3 разных входах** — покажи что шаблон работает с разными параметрами


In [5]:
# TODO: твой код

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langfuse.langchain import CallbackHandler

from langfuse import Langfuse

langfuse = Langfuse()

langfuse_handler = CallbackHandler()
llm = ChatOllama(model="gemma4:e4b", base_url="http://localhost:11434", temperature=1.0)

# 1. Свой промпт с минимум 2 переменными
my_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — преподаватель, который создает тестовые вопросы по теме '{topic}'. "
     "Уровень сложности: {difficulty}. "
     "Сформируй 3 вопроса с 4 вариантами ответа (A-D). "
     "Обязательно укажи правильный ответ и краткое объяснение."
    ),
    ("human",
     "Сгенерируй тест по теме: {topic} с уровнем сложности {difficulty}."
    ),
])

# 2. Собери chain
my_chain = my_prompt | llm | StrOutputParser()

# 3. Запусти минимум 3 раза с разными параметрами
test_inputs = [
    {"topic": "Основы Python", "difficulty": "легкий"},
    {"topic": "Машинное обучение", "difficulty": "средний"},
    {"topic": "История Казахстана", "difficulty": "сложный"},
]

for inp in test_inputs:
    print(f"\n━━━ Вход: {inp}")
    result = my_chain.invoke(inp, config={"callbacks": [langfuse_handler]})
    print(f"Выход: {result}")

langfuse.flush()
print("\n→ Проверь трейсы в http://localhost:3000")



━━━ Вход: {'topic': 'Основы Python', 'difficulty': 'легкий'}
Выход: Приветствую! Как преподаватель, я подготовил для вас небольшой тест по основам Python. Эти вопросы помогут проверить базовое понимание синтаксиса, переменных и основных типов данных.

---

## 📝 Тест: Основы Python (Уровень: Легкий)

### Вопрос 1: Вывод данных
Какая функция в Python используется для вывода текста или переменных в консоль?

A) `output()`
B) `print_data()`
C) `write()`
D) `print()`

***

### Вопрос 2: Типы данных
Какой тип данных в Python используется для представления текстовой информации (например, имени человека или адреса)?

A) `integer` (целое число)
B) `float` (число с плавающей точкой)
C) `string` (строка)
D) `boolean` (логическое значение)

***

### Вопрос 3: Арифметические операции
Если в Python заданы две переменные: `a = 10` и `b = 5`. Какое значение будет у выражения `a // b`? (Оператор `//` используется для целочисленного деления).

A) `2`
B) `2.0`
C) `50`
D) `15`

---
## ✅ Ключ и объяснения

---

## Задание 2 — Свой агент с 2-3 tools (40 баллов)

Собери агента с **2 или 3 инструментами**. Минимум **один** из них должен реально ходить во внешний API (не заглушка).

### Требования

1. **Минимум 2 tools** через `@tool` decorator
2. **Хотя бы один tool** обращается к реальному публичному API (список идей ниже)
3. **Docstrings** у каждого tool чёткие — модель по ним решает когда что вызвать
4. Агентный цикл с `max_iterations` (как в практике) — чтобы не завис
5. Все вызовы с Langfuse callback
6. Протести агента на **5 разных запросах**, из них минимум 2 должны требовать нескольких tool calls подряд

### Идеи tools с бесплатными публичными API

**Простые (не требуют ключей):**

- `get_exchange_rate(from_currency, to_currency)` — курс валют через `https://open.er-api.com/v6/latest/USD` или `https://api.exchangerate-api.com/v4/latest/USD`
- `get_crypto_price(coin)` — цена криптовалюты через CoinGecko: `https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd`
- `get_random_fact(category)` — случайный факт через `https://uselessfacts.jsph.pl/api/v2/facts/random`
- `get_wiki_summary(query)` — поиск в Википедии: `https://en.wikipedia.org/api/rest_v1/page/summary/{query}`
- `get_github_user(username)` — инфа о юзере: `https://api.github.com/users/{username}`
- `get_news_headlines()` — новости через RSS (парсить Казинформ, Tengrinews)

**Локальные (на своей машине):**

- `calculator(expression)` — вычисление математических выражений
- `list_files(path)` — список файлов в папке (ограничь безопасно!)
- `get_system_info()` — инфа о системе через `platform` модуль

### Обязательно в коде

- `max_iterations=5` в цикле агента
- Обработка ошибок в tools — если API упал, вернуть понятное сообщение
- Таймауты у всех `requests.get()` — не больше 10 секунд


In [6]:
# TODO: твой код

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
import requests
import numexpr as ne

# Tool 1 — с реальным API
@tool
def get_exchange_rate(pair: str) -> str:
    """Получает курс валют.

    Используй этот tool, когда нужно узнать курс валют или конвертировать деньги.

    Args:
        pair: строка формата "USD/KZT", "EUR/USD" и т.д.
    """
    try:
        base, target = pair.upper().split("/")

        url = f"https://open.er-api.com/v6/latest/{base}"
        resp = requests.get(url, timeout=10)

        if resp.status_code != 200:
            return "Ошибка API курсов валют"

        data = resp.json()
        rate = data["rates"].get(target)

        if rate is None:
            return f"Не удалось найти курс для {pair}"

        return f"1 {base} = {rate} {target}"

    except Exception as e:
        return f"Ошибка при получении курса: {str(e)}"


# Tool 2 — можно локальный
@tool
def calculator(expression: str) -> str:
    """Вычисляет математические выражения.

    Используй для любых вычислений (сложение, умножение, проценты и т.д.)

    Args:
        expression: строка с выражением, например "100 * 0.9" или "500 + 200"
    """
    try:
        result = ne.evaluate(expression)
        return str(result)

    except Exception as e:
        return f"Ошибка вычисления: {str(e)}"


# Проверяем что tools работают сами по себе
print("Тест tool 1:", get_exchange_rate.invoke({"pair": "USD/KZT"}))
print("Тест tool 2:", calculator.invoke({"expression": "2+2"}))


Тест tool 1: 1 USD = 464.245336 KZT
Тест tool 2: 4


In [7]:
# Агентный цикл с max_iterations

tools = [get_exchange_rate, calculator]
available = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)


def run_agent(user_message: str, max_iterations: int = 5) -> str:
    messages = [HumanMessage(user_message)]

    for i in range(max_iterations):
        ai = llm_with_tools.invoke(
            messages,
            config={"callbacks": [langfuse_handler]},
        )
        messages.append(ai)

        if not ai.tool_calls:
            return ai.content

        print(f"  [iter {i+1}] tools: {[tc['name'] for tc in ai.tool_calls]}")
        for tc in ai.tool_calls:
            result = available[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return "⚠ max_iterations exceeded"


# TODO: 5 тестовых запросов, минимум 2 — с несколькими tool calls
test_queries = [
    # простые (1 tool)
    "Какой курс USD к KZT?",
    "Сколько будет 45 * 12?",
    "Посчитай 1000 - 275",

    # сложные (несколько tool calls)
    "Сколько будет 100 долларов в тенге?",
    "Если у меня 250 долларов, переведи в тенге по текущему курсу и прибавь 5000 тенге",
]

for q in test_queries:
    print(f"\n━━━ Запрос: {q}")
    answer = run_agent(q)
    print(f"\nОТВЕТ: {answer}\n")

langfuse.flush()
print("\n→ Проверь trace-дерево в Langfuse UI")



━━━ Запрос: Какой курс USD к KZT?
  [iter 1] tools: ['get_exchange_rate']

ОТВЕТ: 1 USD = 464.245336 KZT.


━━━ Запрос: Сколько будет 45 * 12?
  [iter 1] tools: ['calculator']

ОТВЕТ: 540


━━━ Запрос: Посчитай 1000 - 275
  [iter 1] tools: ['calculator']

ОТВЕТ: 1000 - 275 = 725


━━━ Запрос: Сколько будет 100 долларов в тенге?
  [iter 1] tools: ['get_exchange_rate']

ОТВЕТ: 100 долларов в тенге составит **46 424,53** тенге (на основе курса 1 USD = 464,245336 KZT).


━━━ Запрос: Если у меня 250 долларов, переведи в тенге по текущему курсу и прибавь 5000 тенге
  [iter 1] tools: ['get_exchange_rate']
  [iter 2] tools: ['calculator']

ОТВЕТ: При текущем курсе (1 USD ≈ 464.25 KZT):

1. **Конвертация долларов:** 250 долларов * 464.245336 = **116 061,33 KZT**
2. **Прибавление тенге:** 116 061,33 KZT + 5 000 KZT = **121 061,33 KZT**

Итого, у вас получится примерно **121 061 тенге**.


→ Проверь trace-дерево в Langfuse UI


### Сюда вставь скриншот trace-дерева из Langfuse

Открой любой из трейсов в Langfuse UI где видно вызовы tools. Сделай скриншот развёрнутого дерева.

```
[langfuse trace tree screenshot]
```


---

## Задание 3 — Теоретические вопросы (20 баллов)

Ответь на 5 вопросов **своими словами**, 2-4 предложения на каждый. Просто "да/нет" не засчитывается.

### Вопрос 1

В чём **ключевое отличие** между прямым вызовом Ollama через `openai` SDK (как в блоке 1 практики) и использованием LangChain? Когда имеет смысл LangChain, а когда хватает голого SDK?

**Ответ:**

Прямой вызов через OpenAI SDK (или Ollama-совместимый API) — это просто отправка prompt → получение ответа, без дополнительной логики. LangChain добавляет уровень абстракции: цепочки, агенты, tools, память и удобную композицию компонентов. Если задача простая (один запрос — один ответ), SDK хватает. LangChain имеет смысл, когда есть сложная логика: несколько шагов, вызовы инструментов, пайплайны или переиспользуемые компоненты.

---

### Вопрос 2

Что такое **observability** в контексте LLM-приложений и зачем она нужна? Приведи 2 конкретных примера ситуаций, когда без трейсов в Langfuse найти проблему было бы **очень** сложно.

**Ответ:**

Observability — это возможность видеть, что происходит внутри LLM-приложения: какие промпты отправляются, какие ответы приходят, какие tools вызываются и в каком порядке. Это нужно для отладки, анализа ошибок и улучшения качества.
Пример 1: агент даёт неправильный ответ — без трейсов непонятно, он сам ошибся или tool вернул плохие данные.
Пример 2: модель не вызывает нужный tool — без логов невозможно понять, проблема в prompt, docstring или самой модели.

---

### Вопрос 3

Объясни **агентный цикл**: что именно происходит между моментом когда пользователь задал вопрос и финальным ответом, если агент использует 2 tools? Почему нужен `max_iterations`?

**Ответ:**

Агентный цикл — это итеративный процесс: модель получает вопрос, решает — ответить сразу или вызвать tool, затем получает результат tool и снова думает, что делать дальше. Это может повторяться несколько раз, пока не будет финального ответа. Если используется 2 tools, агент может вызывать их последовательно (например: получить курс → посчитать сумму). max_iterations нужен, чтобы избежать бесконечного цикла, если модель застрянет или будет постоянно вызывать инструменты.

---

### Вопрос 4

Почему docstring у `@tool`-функции критически важен? Что произойдёт если docstring будет плохим или отсутствовать вообще?

**Ответ:**

Docstring — это главный источник информации для модели о том, что делает tool и когда его использовать. LLM читает описание и на его основе принимает решение о вызове. Если docstring плохой или отсутствует, модель может не вызвать нужный tool или использовать его неправильно. В итоге агент либо даёт неверный ответ, либо вообще игнорирует инструменты.

---

### Вопрос 5

Gemma 4 e4b имеет **42% на Tau2 benchmark** (задачи с tool use). Это означает что примерно каждая вторая сложная задача может выполниться неправильно. Назови **3 способа** компенсировать это ограничение, когда строишь production-приложение на маленькой локальной модели.

**Ответ:**

Первый способ — усиливать промпты (например, явно заставлять использовать tools и не придумывать данные). Второй — добавлять валидацию и постобработку (проверять ответы, пересчитывать или переспрашивать модель). Третий — использовать гибридный подход: маленькая модель для простых задач, а сложные случаи отправлять на более мощную модель или повторять несколько раз (self-consistency).


---

## Чеклист перед сдачей

- [ ] Все ячейки выполнены без ошибок, вывод сохранён в ноутбуке
- [ ] Задание 0: скриншоты `ollama list`, `docker compose ps`, Langfuse UI приложены
- [ ] Задание 1: промпт НЕ про sentiment analysis, минимум 2 переменных, 3 разных теста
- [ ] Задание 2: минимум 2 tools, хотя бы один с реальным API, 5 тестовых запросов, скриншот trace-дерева приложен
- [ ] Задание 3: на все 5 вопросов ответы своими словами, 2-4 предложения на каждый
- [ ] Файл назван `Homework_45_<фамилия>.ipynb`

**Как сдавать:**
Архив `Homework_45_<фамилия>.zip` содержит:
- `Homework_45_<фамилия>.ipynb` (этот ноутбук)
- папку `screenshots/` со всеми скриншотами

Загружай в LMS / присылай куратору по ссылке из чата курса.

**Удачи!** 🚀 Если застрял — пиши в Telegram-группу курса.
